In [1]:
import bw2data, bw2io, bw2calc
from bw_timex import TimexLCA
from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
import numpy as np
from datetime import datetime
import os
import re
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys
sys.path.append('../utils/') 
from elec_builder import *

In [3]:
# activate the bw project
bw2data.projects.set_current("ei311")
#for db in bw2data.databases:
#    print(db, len(bw2data.Database(db)))

### 1. building PV_foreground 
- PV: electricity production, photovoltaic, commercial:::  CN|US

In [ ]:
pv_loc = ['CN', 'US']

xx = build_dynamic_electricity_all(
    locations = pv_loc , 
    pathways = [ "SSP1-VLLO" , "SSP2-M", "SSP5-H" ],
    years = [2030, 2040, 2050], 
    elec_act = 'electricity production, photovoltaic, commercial',
    ref_name = 'market for electricity, PV, low voltage',
    fg_db_name="elec_PV_foreground",
    flush_fg_db = True
)

# not tested: assign_td_from_results_dict(results_dict = xx, elec_td_year=10)

In [ ]:
pv_db = bw2data.Database("elec_PV_foreground")
len(pv_db)

In [ ]:
act_list = list(pv_db)[2:4]
for act in act_list:
    tech_excs = list(act.technosphere())
    print(len(tech_excs))
    exc = tech_excs[0]
    print(exc)

In [ ]:
list(pv_db) 

In [ ]:
rows = []   # collect results for all acts
act_list = list(pv_db)   
for act in act_list: 
    print(act)

    name = act.get("name")
    name_parts = [p.strip() for p in name.split(",")]
    # run static LCI + premise_GWP vs. pGWP100 first: 
    pgwp_fixedco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - fixed-AGWPCO2")
    pgwp_dpco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - dp-AGWPCO2")
    gwp =  ('ecoinvent-3.11', 'IPCC 2021 (incl. biogenic CO2)', 'climate change: total (incl. biogenic CO2, incl. SLCFs)', 'global warming potential (GWP100)')   

    lca_gwp = bw2calc.lca.LCA({act: 1}, method=gwp)
    lca_gwp.lci(); lca_gwp.lcia()
    score_gwp = float(lca_gwp.score)

    lca_fixed = bw2calc.lca.LCA({act: 1}, method=pgwp_fixedco2)
    lca_fixed.lci(); lca_fixed.lcia()
    score_fixedco2 = float(lca_fixed.score)

    lca_dp = bw2calc.lca.LCA({act: 1}, method=pgwp_dpco2)
    lca_dp.lci(); lca_dp.lcia()
    score_dpco2 = float(lca_dp.score)

    print(score_gwp, score_fixedco2, score_dpco2) 

    # ---- store results for this activity ----
    rows.append({
        "Activity": name,                      # index value later
        "gwp100": score_gwp,
        "pGWP100_fixedCO2": score_fixedco2,
        "pGWP100_dpCO2": score_dpco2,
    })



In [ ]:
import pandas as pd

df_scores = pd.DataFrame(rows)
df_scores = df_scores.set_index("Activity")
df_scores

In [ ]:
df_scores.sort_values(by=['Activity']).to_excel("dp-LCI_output/static/PV_CN_US_staticLCI_threeGWP100.xlsx")

### 2. building dpLCI

In [4]:
pv_db = bw2data.Database('elec_PV_foreground')
len(list(pv_db) )

18

In [5]:
database_dates = {
    'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP5-H_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22': datetime.strptime("2050", "%Y"),
    
    "elec_PV_foreground": "dynamic", # flag databases that should be temporally distributed with "dynamic"
}


#### run seperately for each MY, for 2030 & 2050, otherwise memory/storage error

In [6]:
pv_db_2050 = [
    act for act in pv_db 
    if "2050" in str(act.get('name', ''))
]


pv_db_2030 = [
    act for act in pv_db 
    if "2030" in str(act.get('name', ''))
]

(len(pv_db_2050), len(pv_db_2030))

(6, 6)

In [7]:
dp_results = {}
MY  = 2050 # just one year one time, otherwise, storage error

for act in list(pv_db_2050): 

    #### dynamic LCI:: 
    assign_td_from_foreground_db( 
        select_act = act,
        elec_td_year=10,
        resolution="Y",
        kind="uniform",
        fg_db_name="elec_PV_foreground",
        verbose = True
     )

    tlca = run_dp_timex_lca(foreground_act = act ,   
                    pathway = None,
                    year = MY,
                    method = None,
                    database_dates = database_dates, #None not working, has to incl. all 9 background DB ... 
                    temporal_grouping="year", 
                    method_prefix = "Climate Change prospective GWP100",
                    method_suffix = "pGWP100 - fixed-AGWPCO2", 
                    fg_db_name = 'elec_PV_foreground'
     )

    tlca.lci()
    tlca.dynamic_inventory.shape

    # dyn-foreground LCI + static background LCI + pGWP100
    lca_0 = tlca.base_score 
    
    # dpLCI (dyn-foreground LCI + dyn background LCI) +  pGWP100
    tlca.static_lcia()
    lca_1 = tlca.static_score


    act_name = act.get("name")
    dp_results[act_name] = {
        "dyn FG LCI static BG p-LCI, pGWP100-fixedCO2": float(lca_0),   # static BG LCI + dyn FG LCI
        "full dp-LCI, pGWP100-fixedCO2": float(lca_1),      # dyn BG LCI + dyn FG LCI
    }
    '''
    #### now save all dyLCI flows to pandas 
    df = tlca.dynamic_inventory_df

    ##### important to convert flow and act as str to excel 
    df["flow"] = df["flow"].astype(str)
    df["activity"] = df["activity"].astype(str)

    ### export df to dp-LCI_output folder, using the act name as the excel name 
    out_dir = "dp-LCI_output"
    os.makedirs(out_dir, exist_ok=True)    
    raw_name = act["name"]
    safe_name = re.sub(r"[^A-Za-z0-9_\-()]+", "_", raw_name)   # replace spaces/special chars
    
    excel_path = os.path.join(out_dir, f"{safe_name}.xlsx")
    
    df.to_excel(excel_path, index=False)    
    print(f"✔ Exported dynamic inventory DF for '{raw_name}' → {excel_path}")
    '''




out_dir = "dp-LCI_output/PV_dpLCI"
os.makedirs(out_dir, exist_ok=True)

pickle_path = os.path.join(out_dir, "PV_dp_results_MY2050.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(dp_results, f)

print(f"✔ Saved dp_results dictionary → {pickle_path}")

2025-12-06 07:30:22.890 | INFO     | bw_timex.timex_lca:__init__:114 - Initializing TimexLCA object...
2025-12-06 07:30:22.891 | INFO     | bw_timex.timex_lca:__init__:136 - Collecting node infos...


TD applied to 'market for electricity, PV, low voltage, CN, SSP5-H, 2050' → Exchange: 1 kilowatt hour 'electricity production, photovoltaic, commercial' (kilowatt hour, CN, None) to 'market for electricity, PV, low voltage, CN, SSP5-H, 2050' (kWh, CN, None)>
for the activity 'market for electricity, PV, low voltage, CN, SSP5-H, 2050' (kWh, CN, None),it's under SSP-SSP5-H, year-2050 
 we'll use LCIA ('Climate Change prospective GWP100', 'SSP585', 'MY2050', 'pGWP100 - fixed-AGWPCO2')  with background database.datetime = {'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.datetime(2040, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.datetime(2050, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetime.datetime(2040, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22':

2025-12-06 07:33:23.999 | INFO     | bw_timex.timex_lca:build_timeline:216 - No edge filter function provided. Skipping all edges in background databases.
2025-12-06 07:34:28.780 | INFO     | bw_timex.timex_lca:build_timeline:232 - Calculating base LCA...
2025-12-06 07:34:58.455 | INFO     | bw_timex.timex_lca:build_timeline:242 - Creating activity time mapping...
2025-12-06 07:35:01.105 | INFO     | bw_timex.timeline_builder:__init__:99 - Traversing supply chain graph...


Starting graph traversal


2025-12-06 07:35:02.891 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...
2025-12-06 07:35:02.975 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 07:35:02.976 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 07:35:02.978 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 07:35:02.980 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all prov

...

Starting graph traversal


2025-12-06 08:28:48.547 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...
2025-12-06 08:28:48.649 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 08:28:48.649 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 08:28:48.651 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 08:28:48.652 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all prov

Calculation count: 1


2025-12-06 08:29:27.023 | INFO     | bw_timex.timex_lca:lci:360 - Expanding matrices...
2025-12-06 08:29:27.207 | INFO     | bw_timex.timex_lca:lci:379 - Calculating dynamic inventory...


✔ Saved dp_results dictionary → dp-LCI_output/PV_dpLCI/PV_dp_results_MY2050.pkl


In [8]:


dp_results = {}
MY  = 2030 # just one year one time, otherwise, storage error

for act in list(pv_db_2030): 
    print(act)
 
    #### dynamic LCI:: 
    assign_td_from_foreground_db( 
        select_act = act,
        elec_td_year=10,
        resolution="Y",
        kind="uniform",
        fg_db_name="elec_PV_foreground",
        verbose = True
     )

    tlca = run_dp_timex_lca(foreground_act = act ,   
                    pathway = None,
                    year = MY,
                    method = None,
                    database_dates = database_dates, #None not working, has to incl. all 9 background DB ... 
                    temporal_grouping="year", 
                    method_prefix = "Climate Change prospective GWP100",
                    method_suffix = "pGWP100 - fixed-AGWPCO2", 
                    fg_db_name = 'elec_PV_foreground'
     )

    tlca.lci()
    tlca.dynamic_inventory.shape

    # dyn-foreground LCI + static background LCI + pGWP100
    lca_0 = tlca.base_score 
    
    # dpLCI (dyn-foreground LCI + dyn background LCI) +  pGWP100
    tlca.static_lcia()
    lca_1 = tlca.static_score


    act_name = act.get("name")
    dp_results[act_name] = {
        "dyn FG LCI static BG p-LCI, pGWP100-fixedCO2": float(lca_0),   # static BG LCI + dyn FG LCI
        "full dp-LCI, pGWP100-fixedCO2": float(lca_1),      # dyn BG LCI + dyn FG LCI
    }



out_dir = "dp-LCI_output/PV_dpLCI"
os.makedirs(out_dir, exist_ok=True)

pickle_path = os.path.join(out_dir, "PV_dp_results_MY2030.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(dp_results, f)

print(f"✔ Saved dp_results dictionary → {pickle_path}")

2025-12-06 08:30:10.260 | INFO     | bw_timex.timex_lca:__init__:114 - Initializing TimexLCA object...
2025-12-06 08:30:10.261 | INFO     | bw_timex.timex_lca:__init__:136 - Collecting node infos...


'market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030' (kWh, CN, None)
TD applied to 'market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030' → Exchange: 1 kilowatt hour 'electricity production, photovoltaic, commercial' (kilowatt hour, CN, None) to 'market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030' (kWh, CN, None)>
for the activity 'market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030' (kWh, CN, None),it's under SSP-SSP1-VLLO, year-2030 
 we'll use LCIA ('Climate Change prospective GWP100', 'SSP119', 'MY2030', 'pGWP100 - fixed-AGWPCO2')  with background database.datetime = {'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.datetime(2040, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.datetime(2050, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2040 2025-

2025-12-06 08:34:13.943 | INFO     | bw_timex.timex_lca:build_timeline:216 - No edge filter function provided. Skipping all edges in background databases.
2025-12-06 08:39:17.973 | INFO     | bw_timex.timex_lca:build_timeline:232 - Calculating base LCA...
2025-12-06 08:39:43.548 | INFO     | bw_timex.timex_lca:build_timeline:242 - Creating activity time mapping...
2025-12-06 08:39:46.490 | INFO     | bw_timex.timeline_builder:__init__:99 - Traversing supply chain graph...


Starting graph traversal


2025-12-06 08:39:48.333 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


...

Starting graph traversal


2025-12-06 09:43:46.784 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


Calculation count: 1


2025-12-06 09:43:46.937 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 09:43:46.940 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 09:43:46.941 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 09:43:46.943 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-06 09:43:46.944 | INFO     | bw_time

✔ Saved dp_results dictionary → dp-LCI_output/PV_dpLCI/PV_dp_results_MY2030.pkl
